# 🔇 Updated Non-Gunshot Trimmer — Configurable Duration

> Extracts clean background audio clips with impulse rejection.
> **Key upgrade**: Clip duration is fully configurable via `TARGET_MS`.

In [1]:
%pip install -q librosa soundfile pandas numpy tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
import json
import shutil
import random
import re
import IPython.display as ipd

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
print(' All imports loaded successfully.')

 All imports loaded successfully.


c:\ProgramData\anaconda3\envs\guns\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# ============================================================
# CELL 3: CONFIGURATION
# ============================================================

# ==========================================================
#  CLIP DURATION — Must match your Gunshot Trimmer setting!
# ==========================================================
TARGET_MS = 750
# ==========================================================

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'Data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'Data'
OUTPUT_DIR = DATA_DIR / f'TRIMMED_NONGUNSHOTS_{TARGET_MS}MS'

SAMPLE_RATE = 22050
TARGET_SAMPLES = int(SAMPLE_RATE * TARGET_MS / 1000)

CLASS0_HOP_MS = int(TARGET_MS * 0.5)  # 50% overlap, scales with duration
CLASS0_HOP_SAMPLES = int(SAMPLE_RATE * CLASS0_HOP_MS / 1000)

MAX_CLIPS_PER_FILE = 40

CLASS0_DIRS = [DATA_DIR / 'sound']
INCLUDE_AUDIO_FOLDER = True
if INCLUDE_AUDIO_FOLDER:
    CLASS0_DIRS.append(DATA_DIR / 'audio')

# --- Impulse Rejection Thresholds ---
PEAK_SILENCE = 0.003
CREST_IMPULSE = 8.5
PEAK_IMPULSE = 0.1
CENTROID_IMPULSE = 5000.0
ATTACK_RATIO_IMPULSE = 10.0

OVERWRITE = True

print(f'Project Root      : {PROJECT_ROOT}')
print(f'Data Directory    : {DATA_DIR}')
print(f'Output            : {OUTPUT_DIR}')
print(f'Clip Duration     : {TARGET_MS}ms = {TARGET_SAMPLES} samples @ {SAMPLE_RATE}Hz')
print(f'Hop               : {CLASS0_HOP_MS}ms (50% overlap)')
print(f'Max Clips/File    : {MAX_CLIPS_PER_FILE}')
print(f'Source dirs       : {[str(d) for d in CLASS0_DIRS]}')
assert DATA_DIR.exists(), f'❌ Data directory not found: {DATA_DIR}'

Project Root      : C:\order\Desktop\Gun\Data-Cleaner
Data Directory    : C:\order\Desktop\Gun\Data-Cleaner\Data
Output            : C:\order\Desktop\Gun\Data-Cleaner\Data\TRIMMED_NONGUNSHOTS_750MS
Clip Duration     : 750ms = 16537 samples @ 22050Hz
Hop               : 375ms (50% overlap)
Max Clips/File    : 40
Source dirs       : ['C:\\order\\Desktop\\Gun\\Data-Cleaner\\Data\\sound', 'C:\\order\\Desktop\\Gun\\Data-Cleaner\\Data\\audio']


In [10]:
# ============================================================
# CELL 4: Helper Functions
# ============================================================
JUNK_TOKENS = ('__MACOSX',)
SAFE_NAME_RE = re.compile(r'[^A-Za-z0-9._-]+')

def is_junk(path):
    as_str = str(path)
    return any(t in as_str for t in JUNK_TOKENS) or path.name.startswith('._')

def sanitize_name(raw, max_len=120):
    cleaned = SAFE_NAME_RE.sub('_', raw.strip()).strip('._')
    return (cleaned or 'clip')[:max_len]

def collect_wavs(dirs):
    files = []
    for d in dirs:
        if d.exists():
            files.extend([f for f in d.rglob('*.wav') if not is_junk(f)])
    return sorted(files)

def load_audio(path):
    y, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y.astype(np.float32)

def force_exact_length(clip):
    if len(clip) == TARGET_SAMPLES:
        return clip
    if len(clip) > TARGET_SAMPLES:
        return clip[:TARGET_SAMPLES]
    return np.pad(clip, (0, TARGET_SAMPLES - len(clip)), mode='constant')

def normalize_clip(clip):
    clip = clip - np.mean(clip)
    peak = float(np.max(np.abs(clip))) if len(clip) else 0.0
    if peak > 0.999:
        clip = clip / peak * 0.999
    return clip.astype(np.float32)

def write_clip(path, clip):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), clip, SAMPLE_RATE, subtype='PCM_16')
    if not path.exists() or path.stat().st_size <= 44:
        raise IOError(f'Empty/invalid file written: {path}')

print(' Helper functions defined.')

 Helper functions defined.


In [11]:
# ============================================================
# CELL 5: Background Clip Extraction (Sliding Window)
# ============================================================

def extract_background_windows(y):
    if y is None or len(y) < 100:
        return []
    windows = []
    if len(y) < TARGET_SAMPLES:
        clip = force_exact_length(y)
        clip = normalize_clip(clip)
        windows.append((clip, 0, len(y)))
        return windows
    starts = list(range(0, len(y) - TARGET_SAMPLES + 1, CLASS0_HOP_SAMPLES))
    random.shuffle(starts)
    for start_idx in starts:
        end_idx = start_idx + TARGET_SAMPLES
        clip = y[start_idx:end_idx]
        clip = force_exact_length(clip)
        clip = normalize_clip(clip)
        windows.append((clip, start_idx, end_idx))
        if len(windows) >= MAX_CLIPS_PER_FILE:
            break
    return windows

print(' Background extraction function defined.')

 Background extraction function defined.


In [12]:
# ============================================================
# CELL 6: Impulse Rejection Filter
# ============================================================

def is_impulse_free(clip):
    abs_clip = np.abs(clip)
    peak = float(np.max(abs_clip))
    rms = float(np.sqrt(np.mean(np.square(clip))))
    crest = peak / (rms + 1e-12)
    if peak < PEAK_SILENCE:
        return False, 'silence'
    if crest > CREST_IMPULSE and peak > PEAK_IMPULSE:
        return False, 'impulse_detected'
    try:
        centroid = float(np.mean(librosa.feature.spectral_centroid(y=clip, sr=SAMPLE_RATE)[0]))
    except Exception:
        centroid = 0.0
    first_half_energy = float(np.mean(np.square(clip[:len(clip)//2])))
    second_half_energy = float(np.mean(np.square(clip[len(clip)//2:])))
    attack_ratio = first_half_energy / (second_half_energy + 1e-12)
    if centroid > CENTROID_IMPULSE and attack_ratio > ATTACK_RATIO_IMPULSE:
        return False, 'spectral_impulse'
    return True, 'clean'

print(' Impulse rejection filter defined.')

 Impulse rejection filter defined.


In [13]:
# ============================================================
# CELL 7: BUILD PIPELINE
# ============================================================

def build_nongunshot_dataset():
    if OUTPUT_DIR.exists() and OVERWRITE:
        shutil.rmtree(OUTPUT_DIR)
    
    clean_dir = OUTPUT_DIR / 'clean'
    rejected_dir = OUTPUT_DIR / 'rejected'
    reports_dir = OUTPUT_DIR / 'reports'
    clean_dir.mkdir(parents=True, exist_ok=True)
    rejected_dir.mkdir(parents=True, exist_ok=True)
    reports_dir.mkdir(parents=True, exist_ok=True)
    
    source_files = collect_wavs(CLASS0_DIRS)
    print(f'\n{"=" * 65}')
    print(f'NON-GUNSHOT TRIMMER — Starting Build')
    print(f'{"=" * 65}')
    print(f'Source files found  : {len(source_files):,}')
    print(f'Output directory    : {OUTPUT_DIR}')
    print(f'Clip duration       : {TARGET_MS}ms ({TARGET_SAMPLES} samples)')
    print(f'{"=" * 65}\n')
    
    manifest_rows = []
    clean_count = 0
    rejected_count = 0
    clip_index = 0
    
    for src_path in tqdm(source_files, desc='🔇 Extracting background', unit='file'):
        src_stem = sanitize_name(src_path.stem)
        src_parent = sanitize_name(src_path.parent.name)
        source_key = f'{src_parent}_{src_stem}'
        
        try:
            y = load_audio(src_path)
        except Exception:
            continue
        
        windows = extract_background_windows(y)
        for win_idx, (clip, start, end) in enumerate(windows):
            passed, reason = is_impulse_free(clip)
            clip_name = f'{source_key}_win{win_idx:03d}_{clip_index:06d}.wav'
            
            if passed:
                write_clip(clean_dir / clip_name, clip)
                clean_count += 1
            else:
                write_clip(rejected_dir / clip_name, clip)
                rejected_count += 1
            
            manifest_rows.append({
                'filename': clip_name, 'source_file': str(src_path),
                'source_key': source_key, 'start': start, 'end': end,
                'decision': reason, 'clip_duration_ms': TARGET_MS,
            })
            clip_index += 1
    
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(reports_dir / 'manifest.csv', index=False)
    
    print(f'\n{"=" * 65}')
    print(f'BUILD COMPLETE')
    print(f'{"=" * 65}')
    print(f'Clean clips     : {clean_count:,}')
    print(f'Rejected clips  : {rejected_count:,}')
    print(f'Total output    : {clip_index:,}')
    print(f'Output folder   : {OUTPUT_DIR}')
    print(f'{"=" * 65}')
    return manifest_df

manifest_df = build_nongunshot_dataset()


NON-GUNSHOT TRIMMER — Starting Build
Source files found  : 6,230
Output directory    : C:\order\Desktop\Gun\Data-Cleaner\Data\TRIMMED_NONGUNSHOTS_750MS
Clip duration       : 750ms (16537 samples)



🔇 Extracting background: 100%|██████████| 6230/6230 [04:33<00:00, 22.79file/s]



BUILD COMPLETE
Clean clips     : 54,826
Rejected clips  : 19,930
Total output    : 74,756
Output folder   : C:\order\Desktop\Gun\Data-Cleaner\Data\TRIMMED_NONGUNSHOTS_750MS


In [14]:
# ============================================================
# CELL 8: VERIFICATION
# ============================================================
print('Verifying all output files...\n')
clean_files = list((OUTPUT_DIR / 'clean').glob('*.wav'))
bad_files = []
for f in tqdm(clean_files[:500], desc='Checking sizes'):
    y, sr = librosa.load(f, sr=None)
    if len(y) != TARGET_SAMPLES:
        bad_files.append((f.name, len(y)))

if bad_files:
    print(f'\n {len(bad_files)} files have wrong length!')
else:
    print(f'\n Checked {min(500, len(clean_files)):,} files — all exactly {TARGET_SAMPLES} samples ({TARGET_MS}ms).')
print(f'\n Total clean clips: {len(clean_files):,}')

Verifying all output files...



Checking sizes: 100%|██████████| 500/500 [00:00<00:00, 7306.39it/s]


 Checked 500 files — all exactly 16537 samples (750ms).

 Total clean clips: 54,826
